# 🌲 Lab 2: Decision Trees
**BINF 4002 – Machine Learning for Health**

---
## Learning Objectives
1. Understand how decision trees partition feature space using information-theoretic criteria
2. Build, visualize, and manually trace a decision tree
3. Implement tree prediction yourself and validate it
4. Demonstrate and explain overfitting in trees
5. Understand regularization strategies for trees

---
## What is a Decision Tree?

A decision tree is a model that makes predictions by recursively partitioning the input space
into rectangular regions, one split at a time. At each internal node, the algorithm asks:
*which feature, and which threshold on that feature, best separates the classes in my current
subset of training data?* The best split is chosen by maximizing **information gain** — the
reduction in impurity achieved by making the split.

The most common impurity measure for classification is the **Gini impurity**:

$$G(S) = 1 - \sum_{k} p_k^2$$

where $p_k$ is the fraction of samples in set $S$ belonging to class $k$. A pure node (all
one class) has $G=0$; a maximally mixed node has $G=0.5$ for binary classification. The
algorithm greedily picks the split that minimizes the weighted average Gini of the two
child nodes:

$$\text{Gain}(S, j, t) = G(S) - \frac{|S_L|}{|S|}G(S_L) - \frac{|S_R|}{|S|}G(S_R)$$

where $S_L, S_R$ are the subsets falling left and right of threshold $t$ on feature $j$.

This process repeats recursively in each child node until a stopping criterion is met
(maximum depth, minimum samples per leaf, or no further gain). Prediction is then simply:
trace the sample from root to a leaf, and return the majority class in that leaf.

Why do we care about decision trees in healthcare? Three reasons. First, the prediction
rule is completely explicit — you can print it out and hand it to a clinician. Second,
trees naturally handle mixed feature types (continuous, ordinal) without requiring scaling.
Third, and most importantly for this course, understanding trees is the prerequisite for
understanding random forests and gradient boosting, which are among the best-performing
models on structured clinical data.


## Set-up
### Upload data
⚠️ First, you need to upload the pre-processed data from `lab0`. If you have issues with running the first lab, you can also download the data [here](https://drive.google.com/file/d/1mCz8VqpX0F5DzOTnfb5NzpxNAMBrzD-_/view?usp=drive_link).

Once you have downloaded the data locally and started the runtime for this notebook, upload the file via the "Files" menu.

In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

pkl_path = 'processed_data.pkl'
if not os.path.exists(pkl_path):
    raise FileNotFoundError(
        "processed_data.pkl not found. Run Lab 0 first, or download the data "
        "from the link above and upload it here."
    )

with open(pkl_path, 'rb') as f:
    d = pickle.load(f)

# Use the *restricted* hard splits (8 weakest features, 120 train samples)
# This gives realistic AUC rather than a trivially easy problem.
X_train, y_train = d['X_train_hard'], d['y_train_hard']
X_val,   y_val   = d['X_val_hard'],   d['y_val_hard']
X_test,  y_test  = d['X_test_hard'],  d['y_test_hard']
feature_names    = d['feature_names_hard']
class_names      = ['Malignant', 'Benign']

print(f"Train : {X_train.shape}  |  Val : {X_val.shape}  |  Test : {X_test.shape}")
print(f"Features: {feature_names}")


In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score

def print_metrics(y_true, y_pred, y_prob=None, label='Model'):
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred).ravel()
    n = len(y_true)
    sens = tp/(tp+fn); spec = tn/(tn+fp)
    ppv  = tp/(tp+fp); npv  = tn/(tn+fn)
    prev = (tp+fn)/n
    print(f"{'─'*55}")
    print(f"  {label}")
    print(f"{'─'*55}")
    print(f"  Sensitivity  P(ŷ=1|y=1)  = {tp}/{tp+fn} = {sens:.3f}")
    print(f"  Specificity  P(ŷ=0|y=0)  = {tn}/{tn+fp} = {spec:.3f}")
    print(f"  PPV          P(y=1|ŷ=1)  = {tp}/{tp+fp} = {ppv:.3f}")
    print(f"  NPV          P(y=0|ŷ=0)  = {tn}/{tn+fn} = {npv:.3f}")
    print(f"  Prevalence               = {prev:.3f}")
    if y_prob is not None:
        auc = roc_auc_score(y_true, y_prob)
        print(f"  AUC-ROC                  = {auc:.3f}")
    print()


---
## Part 1 — Build and Visualize a Decision Tree

We start with a shallow tree (`max_depth=3`) so the full structure is visible. Before running
the cell, make a prediction: given only 8 weakly-informative features and 120 training
samples, how well do you think this tree will do on the validation set?


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

dt3 = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42)
dt3.fit(X_train, y_train)

plt.figure(figsize=(22, 9))
plot_tree(dt3,
          feature_names=feature_names,
          class_names=class_names,
          filled=True, rounded=True, fontsize=10,
          impurity=True, proportion=False)
plt.title('Decision Tree (max_depth=3) — Restricted Feature Set', fontsize=13)
plt.tight_layout(); plt.show()

print("\nText representation:")
print(export_text(dt3, feature_names=feature_names))

val_preds = dt3.predict(X_val)
val_probs = dt3.predict_proba(X_val)[:, 1]
print_metrics(y_val, val_preds, val_probs, label='Decision Tree (depth=3), Validation')


### Understanding the Tree Output

Each node in the visualization shows:
- **The split condition** (e.g., `mean texture ≤ 0.32`)
- **Gini impurity** of the node — 0.0 is pure, 0.5 is maximally mixed
- **samples** — how many training examples reach this node
- **value** — the count of [malignant, benign] examples at this node
- **class** — the majority class (what would be predicted if we stopped here)

The color intensity reflects purity: darker = more confident. Notice how Gini impurity
decreases as you move from root to leaves — this is the information gain the algorithm is
maximizing at each step.

### 🤔 Reflection 1.1 — Reading the Tree

1. Look at the root node's Gini impurity and the Gini impurities of its two children.
   Compute the information gain of the first split by hand. Does this match what sklearn
   would compute?

2. Find a leaf node with high Gini impurity (close to 0.5). What does this mean clinically?
   Would you trust a model that routes many patients to this leaf?

3. The tree splits only on continuous thresholds. If a feature were ordinal (e.g., tumor
   grade 1–4), how would the tree handle it? What assumption does this implicitly make
   about the ordering?


---
## Part 2 — Implement Tree Prediction Yourself

A decision tree's prediction is a series of comparisons. Implement a function that traces
a *single sample* through the tree using sklearn's internal `tree_` object. This exercise
makes explicit that the "model" is just a lookup table of thresholds, and prediction is
just a traversal — O(depth) operations, no matrix multiplication anywhere.

**Useful attributes of `dt3.tree_`:**
- `.feature[node]` — which feature is split on at this node (-2 = leaf)
- `.threshold[node]` — the threshold value for the split
- `.children_left[node]` / `.children_right[node]` — child node indices
- `.value[node]` — shape `(1, n_classes)` array of class counts at this node


In [ ]:
# ── TODO ──────────────────────────────────────────────────────────────────────
# Implement manual tree traversal for a single sample.

def predict_one_sample_tree(tree_clf, x_sample):
    """
    Manually traverse a fitted DecisionTreeClassifier for a single sample.
    Returns: (predicted_class, list_of_decision_strings)
    """
    tree = tree_clf.tree_
    node = 0       # start at root
    decisions = []

    while tree.feature[node] != -2:   # -2 means leaf
        feat_idx   = tree.feature[node]
        threshold  = tree.threshold[node]
        feat_name  = feature_names[feat_idx]
        feat_value = x_sample[feat_idx]

        # YOUR CODE HERE
        # Append a human-readable decision string to `decisions`
        # Then update `node` to the correct child
        pass

    # At a leaf — majority class
    class_counts = tree.value[node][0]
    predicted_class = int(np.argmax(class_counts))
    return predicted_class, decisions


# Test on a validation sample
sample_idx = 3
pred_class, path = predict_one_sample_tree(dt3, X_val[sample_idx])
print(f"Sample {sample_idx}: True label = {class_names[y_val[sample_idx]]}")
print(f"Manual prediction  : {class_names[pred_class]}")
print(f"Sklearn prediction : {class_names[dt3.predict(X_val[[sample_idx]])[0]]}")
print("\nDecision path:")
for step in path:
    print(f"  {step}")


In [ ]:
# ── Sanity check ──────────────────────────────────────────────────────────────
# Run on all validation samples and compare to sklearn
manual_preds = np.array([predict_one_sample_tree(dt3, x)[0] for x in X_val])
sklearn_preds = dt3.predict(X_val)

n_agree = (manual_preds == sklearn_preds).sum()
assert n_agree == len(X_val), (
    f"Manual predictions disagree with sklearn on {len(X_val)-n_agree} samples. "
    "Check your left/right child logic and the leaf condition."
)
print(f"✓ Manual traversal matches sklearn on all {len(X_val)} validation samples.")


### 🤔 Reflection 2.1 — Trees as Explicit Rules

1. Your implementation is a while loop with a handful of comparisons. Compare this to
   logistic regression's forward pass (a dot product + sigmoid). Which is more computationally
   expensive per sample? Which is easier for a non-technical clinician to audit?

2. You ran the tree manually on a single sample. Could you do this for 200 trees?

3. How "explainable" is a decision tree? What does "explainability" or "interpretability" mean for this kind of model?

3. Decision trees split on thresholds of continuous features. This means the decision
   boundary is always **axis-aligned** (perpendicular to a feature axis). Draw a 2D
   example where this makes trees fundamentally not as appropriate logistic regression.
   Then draw one where trees are more appropriate.
   Which is more expressive: A decision tree or a logisitic regression model?


---
## Part 3 — Overfitting: A Practical Demonstration

Decision trees can memorize training data perfectly — a critical failure mode in clinical
settings where we need the model to *generalize* to new patients, not recall training cases.
With only 120 training examples and 8 features, overfitting happens very quickly.

Before running: at what depth do you expect the training accuracy to reach 100%? At what
depth do you expect validation accuracy to peak?


In [ ]:
depths = list(range(1, 25))
train_accs, val_accs, train_aucs, val_aucs = [], [], [], []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(dt.score(X_train, y_train))
    val_accs.append(dt.score(X_val, y_val))
    train_aucs.append(roc_auc_score(y_train, dt.predict_proba(X_train)[:,1]))
    val_aucs.append(roc_auc_score(y_val,   dt.predict_proba(X_val)[:,1]))

best_depth = depths[np.argmax(val_accs)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, train_vals, val_vals, ylabel in zip(
    axes,
    ['Accuracy', 'AUC-ROC'],
    [train_accs, train_aucs],
    [val_accs, val_aucs],
    ['Accuracy', 'AUC-ROC']
):
    ax.plot(depths, train_vals, 'o-', color='#3498db', lw=2, label='Train')
    ax.plot(depths, val_vals,   's-', color='#e74c3c', lw=2, label='Val')
    ax.axvline(best_depth, color='grey', linestyle='--', alpha=0.7,
               label=f'Best depth = {best_depth}')
    ax.set_xlabel('Max depth'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} vs. Tree Depth (n_train=120)')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f"Best depth (by val accuracy): {best_depth}")
print(f"Train accuracy at depth {best_depth}: {train_accs[best_depth-1]:.3f}")
print(f"Val   accuracy at depth {best_depth}: {val_accs[best_depth-1]:.3f}")
print(f"Train accuracy at depth 20       : {train_accs[19]:.3f}")
print(f"Val   accuracy at depth 20       : {val_accs[19]:.3f}")


### Why Does a Deep Tree Overfit?

A tree with no depth limit will keep splitting until every leaf contains exactly one
training sample, achieving 100% training accuracy but generalizing no better than a
lookup table. The fundamental quantity is the **number of leaves**, which upper-bounds
the number of training samples a tree can perfectly classify: a tree with $L$ leaves can
perfectly fit any dataset with $L$ distinct samples.

For $n$ training samples, a complete binary tree reaches this at depth $\lceil \log_2 n \rceil$
— for $n=120$, that's depth 7. This gives us a concrete, calibrated intuition: trees deeper
than about $\log_2(n)$ are almost always overfitting.

Given our discussion about memorization vs. interpolation, what might you suspect about ways we could make a decision tree less likely to overfit?


In [ ]:
# How does training set size affect the overfitting depth?
# Vary n_train and see at what depth overfitting begins.

fig, ax = plt.subplots(figsize=(11, 5))

for n_train, color in [(5, '#e74c3c'), (60, '#3498db'), (120, '#27ae60')]:
    idx = np.random.RandomState(42).choice(len(X_train), min(n_train, len(X_train)), replace=False)
    Xt, yt = X_train[idx], y_train[idx]
    v_accs = []
    for depth in depths:
        dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
        dt.fit(Xt, yt)
        v_accs.append(dt.score(X_val, y_val))
    ax.plot(depths, v_accs, 'o-', lw=2, label=f'n_train={n_train}', color=color)

ax.axhline(0.5, color='grey', linestyle=':', alpha=0.5, label='Chance')
ax.set_xlabel('Max depth'); ax.set_ylabel('Val Accuracy')
ax.set_title('Validation Accuracy vs. Depth for Different Training Set Sizes')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


### 🤔 Reflection 3.1 — Overfitting and Sample Size

1. At what depth does the tree achieve 100% training accuracy? Compare to $\lceil\log_2(120)\rceil$.
   Does the theory match the experiment? Why might they differ slightly?

2. The validation accuracy peaks and then decreases. Write a one-sentence explanation
   targeted at a clinician: why does giving a model "more flexibility" sometimes make it
   *worse* at predicting new patients?

3. Look at the sample-size experiment. How does the optimal depth change as $n_{\text{train}}$
   grows? Does this match the $\log_2(n)$ heuristic? What does this imply about how
   you should tune max_depth relative to dataset size?

4. **Clinical framing**: Suppose a colleague trains a decision tree on 50 patient records
   and shows you 100% training accuracy. They propose deploying it. Write a two-sentence
   response explaining why this should concern you, without using the word "overfitting."


---
## Part 4 — Regularization Strategies for Trees

The depth sweep showed overfitting. Rather than using cross-validation to pick the best
depth (which we'll do), there are several regularization knobs in sklearn's
`DecisionTreeClassifier` that constrain tree complexity. Understanding them matters because
they reappear as hyperparameters in random forests and XGBoost.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Compare regularization strategies on the training set via 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

configs = {
    'Depth=3, no other constraints':
        DecisionTreeClassifier(max_depth=3, random_state=42),
    'Depth=5, no other constraints':
        DecisionTreeClassifier(max_depth=5, random_state=42),
    'Depth=5, min_samples_leaf=10':
        DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42),
    'Depth=5, min_impurity_decrease=0.01':
        DecisionTreeClassifier(max_depth=5, min_impurity_decrease=0.01, random_state=42),
    'No depth limit':
        DecisionTreeClassifier(random_state=42),
}

print(f"{'Configuration':<45} {'CV AUC (mean±std)'}")
print('─' * 65)
cv_results = {}
for name, clf in configs.items():
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='roc_auc')
    cv_results[name] = scores
    print(f"{name:<45} {scores.mean():.3f} ± {scores.std():.3f}")


In [ ]:
# Visualize the CV distributions
fig, ax = plt.subplots(figsize=(10, 5))
labels = list(cv_results.keys())
data   = [cv_results[k] for k in labels]
short_labels = [l.split(',')[0] for l in labels]

bp = ax.boxplot(data, patch_artist=True, notch=False, vert=True)
colors = ['#3498db','#e74c3c','#27ae60','#9b59b6','#f39c12']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)

ax.set_xticklabels(short_labels, rotation=15, ha='right')
ax.set_ylabel('CV AUC-ROC')
ax.set_title('5-Fold CV AUC for Different Regularization Strategies')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


### 🤔 Reflection 4.1 — Choosing Regularization

1. `min_samples_leaf` requires each leaf to contain at least $k$ training samples.
   Explain geometrically what this does to the decision boundary. How does it relate
   to the overfitting mechanism you diagnosed in Part 3? How does it relate to our in-class discussion on memorization vs. interpolation?

2. `min_impurity_decrease` prevents a split unless it reduces Gini by at least $\delta$.
   This is a different kind of constraint than depth — it is *adaptive* rather than global.
   In what situation would this be preferable to setting max_depth?

3. Look at the CV standard deviation across the five configurations. The no-depth-limit
   tree probably has high variance. Why? What does high variance in CV scores imply about
   deploying that model in a new hospital?

4. **Preview**: None of these regularization tricks fully solve the overfitting
   problem for a single tree. In a later lab, we'll see that ensembles — particularly
   random forests and gradient boosting — address this more fundamentally. Before you
   get there: hypothesize *how* averaging 200 slightly different trees might produce
   better generalization than any one of them.


---
## 🧠 Final Reflection — Decision Trees as a Foundation

Before moving on, answer these:

1. **Bias-variance framing**: A fully grown tree has low bias and high variance. A stump
   (depth=1) has high bias and low variance. Draw the bias-variance tradeoff curve for
   decision trees as a function of max_depth. Where does the optimal operating point land
   in our experiment?

2. **The limits of a single tree**: Even with optimal regularization, a single decision
   tree on our hard dataset achieves AUC ~0.78–0.83. What are the two fundamental reasons
   a single tree is limited? (Hint: one is about the partition it can represent, one is
   about sensitivity to training data.)

3. **Clinical audit**: A hospital wants to use a decision tree to decide which patients
   get expedited biopsy referrals. What are the two main arguments *for* using a tree
   over logistic regression in this context? What are the two main arguments *against*?
